In [1]:
import pandas as pd
import min_features, daily_return
import importlib
importlib.reload(min_features)
importlib.reload(daily_return)

perf_df = pd.read_csv("daily_w_prob.csv")
perf_df['Date'] = perf_df['test_start']
returns = [1, 2, 3, 5, 10]
df_daily = daily_return.pull_daily('QQQ', returns) 
return_cols = df_daily.columns[df_daily.columns.str.contains("Return_")].to_list()
df_returns = df_daily[['Date'] + return_cols][(df_daily['Date'] < '2025-12-20') & (df_daily['Date'] > '2025-01-01')].copy()

In [2]:
import numpy as np
import pandas as pd

def add_streak_cols(df, ret_col, *, date_col="Date"):
    d = df[[date_col, "Close", ret_col]].sort_values(date_col).copy()

    s = d[ret_col].astype("int8")
    grp = s.ne(s.shift()).cumsum()
    streak_len = s.groupby(grp).cumcount() + 1

    d["streak"] = streak_len.where(s.eq(1), -streak_len).astype("int32")
    d["streak_lag1"] = d["streak"].shift(1).fillna(0).astype("Int64")
    return d

def streak_perf_tables(df_daily, perf_df, returns, *, test_days=1):
    out_by_r = {}
    side_by_r = {}

    base = df_daily[["Date", "Close"] + [f"Return_{r}" for r in returns]].copy()

    for r in returns:
        ret_col = f"Return_{r}"

        # streaks for this horizon
        df_r = add_streak_cols(base, ret_col)

        # merge with performance rows for this horizon/test_days
        perf_r = perf_df[(perf_df["horizon"] == r) & (perf_df["test_days"] == test_days)]
        dfm = df_r.merge(perf_r, on="Date", how="inner")

        gcols = ["model", "train_years", "feature_set"]

        # 1) Accuracy by streak_lag1
        acc_piv = dfm.pivot_table(
            index=gcols,
            columns="streak_lag1",
            values="acc",
            aggfunc="mean",
        )

        # 2) Count by streak_lag1
        cnt_piv = dfm.pivot_table(
            index=gcols,
            columns="streak_lag1",
            values="acc",
            aggfunc="size",
        )

        out = pd.concat({"acc": acc_piv, "count": cnt_piv}, axis=1)

        # sort streak columns (negative first, then 0, then positive)
        out = out.reindex(
            columns=sorted(out.columns, key=lambda x: (x[1] >= 0, x[1])),
        )

        out_by_r[r] = out

        # pos/neg summary
        df2 = dfm.copy()
        df2["side"] = np.where(df2["streak"] > 0, "pos", "neg")

        side_perf = (
            df2.groupby(gcols + ["side"], sort=False)
               .agg(n=("acc", "size"), acc=("acc", "mean"))
               .reset_index()
        )

        side_wide = (
            side_perf.pivot(index=gcols, columns="side", values=["acc", "n"])
                     .round(2)
        )

        side_by_r[r] = side_wide

    # optional: one combined object with horizon as an extra index level
    all_out = pd.concat(out_by_r, names=["horizon"])
    all_side = pd.concat(side_by_r, names=["horizon"])

    return out_by_r, side_by_r, all_out, all_side

returns = [2, 5, 10]
out_by_r, side_by_r, all_out, all_side = streak_perf_tables(df_daily, perf_df, returns, test_days=1)

# example: horizon 10 tables
out_by_r[10]
side_by_r[10]

# or combined across horizons
#all_out
all_side


acc           n       
side                                            neg   pos   neg    pos
horizon model         train_years feature_set                         
2       random_forest 4           d-no-skew    0.48  0.74  95.0  133.0
                                  d-no-vix     0.51  0.74  95.0  133.0
                                  daily        0.49  0.71  95.0  133.0
                      6           d-no-skew    0.39  0.82  95.0  133.0
                                  d-no-vix     0.40  0.81  95.0  133.0
                                  daily        0.41  0.81  95.0  133.0
        xgboost       4           d-no-skew    0.52  0.70  95.0  133.0
                                  d-no-vix     0.55  0.70  95.0  133.0
                                  daily        0.51  0.68  95.0  133.0
                      6           d-no-skew    0.49  0.74  95.0  133.0
                                  d-no-vix     0.48  0.77  95.0  133.0
                                  daily        0.49  0.74  95.0  133.0
5       random_forest 4           d-no-skew    0.71  0.88  83.0  145.0
                                  d-no-vix     0.75  0.88  83.0  145.0
                                  daily        0.73  0.88  83.0  145.0
                      6           d-no-skew    0.61  0.92  83.0  145.0
                                  d-no-vix     0.58  0.91  83.0  145.0
                                  daily        0.59  0.92  83.0  145.0
        xgboost       4           d-no-skew    0.59  0.86  83.0  145.0
                                  d-no-vix     0.60  0.81  83.0  145.0
                                  daily        0.61  0.84  83.0  145.0
                      6           d-no-skew    0.51  0.89  83.0  145.0
                                  d-no-vix     0.53  0.87  83.0  145.0
                                  daily        0.52  0.88  83.0  145.0
10      random_forest 4           d-no-skew    0.79  0.91  80.0  148.0
                                  d-no-vix     0.78  0.90  80.0  148.0
                                  daily        0.81  0.91  80.0  148.0
                      6           d-no-skew    0.75  0.95  80.0  148.0
                                  d-no-vix     0.75  0.94  80.0  148.0
                                  daily        0.72  0.93  80.0  148.0
        xgboost       4           d-no-skew    0.76  0.92  80.0  148.0
                                  d-no-vix     0.76  0.91  80.0  148.0
                                  daily        0.72  0.91  80.0  148.0
                      6           d-no-skew    0.71  0.93  80.0  148.0
                                  d-no-vix     0.72  0.93  80.0  148.0
                                  daily        0.68  0.92  80.0  148.0

In [3]:
import numpy as np
import pandas as pd

def flip_bucket_tables_multi(
    df_daily,
    perf_df,
    returns,
    *,
    K=3,
    test_days=1,
    date_col="Date",
    close_col="Close",
    streak_col="streak_lag1",
    w=None,
):
    """
    For each horizon r:
      - builds streak_lag1 from Return_r (via day-to-day streak logic)
      - merges into perf_df rows for (horizon=r, test_days)
      - buckets streak_lag1 into [-K..K] plus tails as +/- (K+1) labeled "3+"
      - computes acc/n wide table and weighted balanced-accuracy score (wba)

    Returns:
      flip_by_r: dict[r] -> flip_wide (MultiIndex columns)
      rf_sorted_by_r: dict[r] -> sorted flip_wide by wba
      all_flip: concat of flip_wide with horizon as index level
    """
    if w is None:
        # weights: ±1 -> 2.0, ±2 -> 1.5, ±3 -> 1.25, ±3+ -> 1.0
        w = {1: 2.0, 2: 1.5, 3: 1.25, "3+": 1.0}

    max_score = sum(w.values())
    gcols = ["model", "train_years", "feature_set"]

    # --- helper: compute streak + streak_lag1 for a given Return_r ---
    def _add_streak(df_base, ret_col):
        d = df_base[[date_col, close_col, ret_col]].sort_values(date_col).copy()
        s = d[ret_col].astype("int8")
        grp = s.ne(s.shift()).cumsum()
        streak_len = s.groupby(grp).cumcount() + 1
        d["streak"] = streak_len.where(s.eq(1), -streak_len).astype("int32")
        d["streak_lag1"] = d["streak"].shift(1).fillna(0).astype("Int64")
        return d

    base = df_daily[[date_col, close_col] + [f"Return_{r}" for r in returns]].copy()

    flip_by_r = {}
    rf_sorted_by_r = {}

    for r in returns:
        ret_col = f"Return_{r}"

        # build streak_lag1
        df_r = _add_streak(base, ret_col)

        # merge with perf (this gives you acc/model/train_years/feature_set/etc)
        perf_r = perf_df[(perf_df["horizon"] == r) & (perf_df["test_days"] == test_days)]
        d = df_r.merge(perf_r, on=date_col, how="inner")

        # --- bucket streak_lag1 into [-K..K] plus tails as +/- (K+1) ---
        d["streak_bucket"] = d[streak_col].clip(lower=-K, upper=K)
        d.loc[d[streak_col] < -K, "streak_bucket"] = -(K + 1)
        d.loc[d[streak_col] >  K, "streak_bucket"] =  (K + 1)

        flip_perf = (
            d.groupby(gcols + ["streak_bucket"], sort=False)
             .agg(n=("acc", "size"), acc=("acc", "mean"))
        )

        flip_wide = pd.concat(
            {"acc": flip_perf["acc"].unstack("streak_bucket"),
             "n":   flip_perf["n"].unstack("streak_bucket")},
            axis=1
        )

        # enforce column order: +1/-1, +2/-2, +3/-3, 3+/-3+
        ordered_cols = []
        for k in [1, 2, 3, "3+"]:
            pb = (K + 1) if k == "3+" else k
            nb = -(K + 1) if k == "3+" else -k
            ordered_cols += [("acc", pb), ("acc", nb), ("n", pb), ("n", nb)]
        flip_wide = flip_wide.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))

        # relabel buckets to strings ("3+", "-3+", etc.)
        rename_cols = []
        for metric, b in flip_wide.columns:
            if b == (K + 1): lab = "3+"
            elif b == -(K + 1): lab = "-3+"
            else: lab = str(b)
            rename_cols.append((metric, lab))
        flip_wide.columns = pd.MultiIndex.from_tuples(rename_cols)

        # --- per-pair bal_acc and weighted score ---
        def _pair_bal(pos_lab, neg_lab):
            return (flip_wide[("acc", pos_lab)] + flip_wide[("acc", neg_lab)]) / 2

        flip_wide[("bal_acc_pair", "1")]  = _pair_bal("1",  "-1")
        flip_wide[("bal_acc_pair", "2")]  = _pair_bal("2",  "-2")
        flip_wide[("bal_acc_pair", "3")]  = _pair_bal("3",  "-3")
        flip_wide[("bal_acc_pair", "3+")] = _pair_bal("3+", "-3+")

        # weighted SUM, normalized by max_score (your current behavior)
        flip_wide[("wba", "")] = (
            w[1]    * flip_wide[("bal_acc_pair", "1")] +
            w[2]    * flip_wide[("bal_acc_pair", "2")] +
            w[3]    * flip_wide[("bal_acc_pair", "3")] +
            w["3+"] * flip_wide[("bal_acc_pair", "3+")]
        ) / max_score

        flip_wide[("wba", "")] = flip_wide[("wba", "")].round(2)

        flip_by_r[r] = flip_wide
        rf_sorted_by_r[r] = flip_wide.sort_values(by=("wba", ""), ascending=False).round(2)

    all_flip = pd.concat(flip_by_r, names=["horizon"])
    return flip_by_r, rf_sorted_by_r, all_flip

returns = [2, 5, 10]
flip_by_r, rf_sorted_by_r, all_flip = flip_bucket_tables_multi(
    df_daily=df_daily,
    perf_df=perf_df,
    returns=returns,
    K=3,
    test_days=1,
)

rf_sorted_by_r[10]     # horizon 10 sorted table
flip_by_r[5]           # horizon 5 unsorted table
all_flip.round(2)               # all horizons stacked with "horizon" index level


acc         n       acc        \
                                                  1    -1   1  -1     2    -2   
horizon model         train_years feature_set                                   
2       xgboost       4           daily        0.67  0.61  36  36  0.57  0.58   
                                  d-no-skew    0.58  0.61  36  36  0.70  0.62   
                                  d-no-vix     0.67  0.53  36  36  0.67  0.67   
                      6           daily        0.69  0.67  36  36  0.60  0.71   
                                  d-no-skew    0.67  0.67  36  36  0.63  0.58   
                                  d-no-vix     0.78  0.67  36  36  0.60  0.67   
        random_forest 4           daily        0.75  0.61  36  36  0.60  0.54   
                                  d-no-skew    0.75  0.64  36  36  0.60  0.50   
                                  d-no-vix     0.81  0.64  36  36  0.53  0.62   
                      6           daily        0.78  0.56  36  36  0.67  0.62   
                                  d-no-skew    0.72  0.64  36  36  0.63  0.54   
                                  d-no-vix     0.83  0.61  36  36  0.67  0.50   
5       xgboost       4           daily        0.72  0.41  18  17  0.93  0.77   
                                  d-no-skew    0.83  0.59  18  17  0.93  0.85   
                                  d-no-vix     0.72  0.41  18  17  0.87  0.85   
                      6           daily        0.72  0.53  18  17  0.93  0.77   
                                  d-no-skew    0.78  0.47  18  17  1.00  0.77   
                                  d-no-vix     0.67  0.47  18  17  0.87  0.85   
        random_forest 4           daily        0.78  0.59  18  17  1.00  1.00   
                                  d-no-skew    0.78  0.65  18  17  1.00  1.00   
                                  d-no-vix     0.78  0.65  18  17  1.00  1.00   
                      6           daily        0.78  0.47  18  17  0.93  0.85   
                                  d-no-skew    0.78  0.53  18  17  1.00  0.85   
                                  d-no-vix     0.72  0.41  18  17  0.93  0.85   
10      xgboost       4           daily        0.64  0.40  11  10  0.90  0.75   
                                  d-no-skew    0.82  0.40  11  10  1.00  0.88   
                                  d-no-vix     0.45  0.40  11  10  0.90  0.75   
                      6           daily        0.55  0.60  11  10  0.90  0.50   
                                  d-no-skew    0.73  0.40  11  10  0.90  0.62   
                                  d-no-vix     0.73  0.50  11  10  0.90  0.50   
        random_forest 4           daily        0.73  0.50  11  10  0.90  0.75   
                                  d-no-skew    0.73  0.40  11  10  1.00  0.75   
                                  d-no-vix     0.55  0.40  11  10  1.00  0.75   
                      6           daily        0.64  0.40  11  10  0.90  0.62   
                                  d-no-skew    0.82  0.30  11  10  1.00  0.62   
                                  d-no-vix     0.73  0.40  11  10  0.90  0.62   

                                                n       acc        ...   n  \
                                                2  -2     3    -3  ...  -3   
horizon model         train_years feature_set                      ...       
2       xgboost       4           daily        30  24  0.68  0.64  ...  14   
                                  d-no-skew    30  24  0.79  0.57  ...  14   
                                  d-no-vix     30  24  0.74  0.57  ...  14   
                      6           daily        30  24  0.79  0.43  ...  14   
                                  d-no-skew    30  24  0.84  0.43  ...  14   
                                  d-no-vix     30  24  0.84  0.29  ...  14   
        random_forest 4           daily        30  24  0.84  0.36  ...  14   
                                  d-no-skew    30  24  0.84  0.57  ...  14   
                              

In [6]:
perf_df

,run,model,test_days,pred,acc,test_n,test_pos_n,test_neg_n,test_pos_frac,test_neg_frac,...,train_start,train_end,test_start,test_end,train_years,horizon_days,n_features,feature_set,horizon,Date
0,1,xgboost,1,1.00,1.0,1,1,0,1.0,0.0,...,2022-01-12,2025-12-18,2025-12-19,2025-12-19,4,2,82,daily,2,2025-12-19
1,1,random_forest,1,0.65,1.0,1,1,0,1.0,0.0,...,2022-01-12,2025-12-18,2025-12-19,2025-12-19,4,2,82,daily,2,2025-12-19
2,2,xgboost,1,0.85,1.0,1,1,0,1.0,0.0,...,2022-01-11,2025-12-17,2025-12-18,2025-12-18,4,2,82,daily,2,2025-12-18
3,2,random_forest,1,0.45,0.0,1,1,0,1.0,0.0,...,2022-01-11,2025-12-17,2025-12-18,2025-12-18,4,2,82,daily,2,2025-12-18
4,3,xgboost,1,0.00,0.0,1,1,0,1.0,0.0,...,2022-01-10,2025-12-16,2025-12-17,2025-12-17,4,2,82,daily,2,2025-12-17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8203,226,random_forest,1,0.40,0.0,1,1,0,1.0,0.0,...,2019-03-06,2025-01-24,2025-01-27,2025-01-27,6,10,76,d-no-vix,10,2025-01-27
8204,227,xgboost,1,0.00,1.0,1,0,1,0.0,1.0,...,2019-03-05,2025-01-23,2025-01-24,2025-01-24,6,10,76,d-no-vix,10,2025-01-24
8205,227,random_forest,1,0.20,1.0,1,0,1,0.0,1.0,...,2019-03-05,2025-01-23,2025-01-24,2025-01-24,6,10,76,d-no-vix,10,2025-01-24
8206,228,xgboost,1,0.00,1.0,1,0,1,0.0,1.0,...,2019-03-04,2025-01-22,2025-01-23,2025-01-23,6,10,76,d-no-vix,10,2025-01-23


In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, log_loss, matthews_corrcoef

gcols = ["horizon", "model", "train_years", "feature_set"]

# expects:
# - results_df["pred"] = P(y=1)
# - results_df["y_true"] (or change y_col below)
# - results_df["acc"] = 1/0 correctness (not used for Brier/MCC/logloss)
y_col = "test_pos_n"   # <-- change if your actual label column is named differently
p_col = "pred"

def _clip01(p, eps=1e-15):
    p = np.asarray(p, dtype=float)
    return np.clip(p, eps, 1 - eps)

def _metrics(g: pd.DataFrame) -> pd.Series:
    y = g[y_col].astype(int).to_numpy()
    p = _clip01(g[p_col].to_numpy())

    # hard preds at 0.5 for MCC
    yhat = (p >= 0.5).astype(int)

    # coverage masks for 0.6/0.4 rule
    sel = (p >= 0.6) | (p <= 0.4)

    # brier / logloss
    brier = brier_score_loss(y, p)
    ll = log_loss(y, p)  # binary log loss

    # MCC (only meaningful if both classes appear in y and yhat)
    mcc = np.nan
    if (np.unique(y).size > 1) and (np.unique(yhat).size > 1):
        mcc = matthews_corrcoef(y, yhat)

    # conditional accuracy when confident
    cov = float(sel.mean())
    acc_conf = float((yhat[sel] == y[sel]).mean()) if sel.any() else np.nan

    # also useful: confident accuracy vs overall accuracy
    acc_all = float((yhat == y).mean())

    return pd.Series({
        "n": len(g),
        "pos_rate": float(y.mean()),
        "brier": float(brier),
        "log_loss": float(ll),
        "mcc": float(mcc),
        "acc_all_0p5": acc_all,
        "cov_pge0p6_ple0p4": cov,
        "acc_pge0p6_ple0p4": acc_conf,
        "n_pge0p6_ple0p4": int(sel.sum()),
    })

metrics_df = (
    perf_df
    .dropna(subset=[y_col, p_col])
    .groupby(gcols, sort=False)
    .apply(_metrics, include_groups=False)
    .reset_index()
    .sort_values(["horizon", "mcc", "brier"], ascending=[True, False, True])
)

metrics_df


,horizon,model,train_years,feature_set,n,pos_rate,brier,log_loss,mcc,acc_all_0p5,cov_pge0p6_ple0p4,acc_pge0p6_ple0p4,n_pge0p6_ple0p4
26,2,xgboost,6,d-no-vix,228.0,0.583333,0.278761,3.195614,0.270499,0.653509,0.916667,0.665072,209.0
24,2,xgboost,4,d-no-vix,228.0,0.583333,0.273816,3.439904,0.261932,0.644737,0.921053,0.647619,210.0
14,2,xgboost,6,d-no-skew,228.0,0.583333,0.283596,2.935039,0.254762,0.644737,0.951754,0.640553,217.0
2,2,xgboost,6,daily,228.0,0.583333,0.278344,3.323941,0.234404,0.635965,0.903509,0.645631,206.0
15,2,random_forest,6,d-no-skew,228.0,0.583333,0.227336,0.646988,0.229107,0.640351,0.631579,0.694444,144.0
25,2,random_forest,4,d-no-vix,228.0,0.583333,0.227357,0.646523,0.226871,0.635965,0.618421,0.680851,141.0
13,2,random_forest,4,d-no-skew,228.0,0.583333,0.231480,0.656266,0.224385,0.635965,0.583333,0.699248,133.0
12,2,xgboost,4,d-no-skew,228.0,0.583333,0.282643,2.908599,0.221313,0.627193,0.929825,0.636792,212.0
27,2,random_forest,6,d-no-vix,228.0,0.583333,0.225965,0.644122,0.207558,0.631579,0.653509,0.704698,149.0
3,2,random_forest,6,daily,228.0,0.583333,0.227226,0.646786,0.195820,0.627193,0.644737,0.693878,147.0


In [19]:
def calibration_table(g, n_bins=8):
    y = g[y_col].astype(int).to_numpy()
    p = np.clip(g[p_col].to_numpy(dtype=float), 1e-15, 1-1e-15)

    # quantile bins (stable counts); duplicates="drop" prevents errors if probs repeat
    bins = pd.qcut(p, q=n_bins, duplicates="drop")

    out = (
        pd.DataFrame({"bin": bins, "y": y, "p": p})
        .groupby("bin", observed=True)
        .agg(
            n=("y", "size"),
            p_mean=("p", "mean"),   # predicted probability avg
            y_rate=("y", "mean"),   # observed frequency
            p_min=("p", "min"),
            p_max=("p", "max"),
        )
        .reset_index(drop=True)
        .sort_values("p_mean")
    )
    return out

cols = ['horizon']
# example: get calibration table for ONE group
key = (2)  # change
g = perf_df.set_index(cols).loc[key].reset_index()
calib_tbl = calibration_table(g, n_bins=10)
round(calib_tbl,2)

,n,p_mean,y_rate,p_min,p_max
0,317,0.09,0.43,0.00,0.20
1,334,0.35,0.39,0.25,0.40
2,184,0.45,0.46,0.45,0.45
3,447,0.52,0.53,0.50,0.55
4,235,0.60,0.69,0.60,0.60
5,228,0.65,0.66,0.65,0.65
6,194,0.70,0.67,0.70,0.70
7,272,0.79,0.71,0.75,0.85
8,525,0.97,0.71,0.90,1.00


In [ ]:
def calibration_table(g, n_bins=8):
    y = g[y_col].astype(int).to_numpy()
    p = np.clip(g[p_col].to_numpy(dtype=float), 1e-15, 1-1e-15)

    # quantile bins (stable counts); duplicates="drop" prevents errors if probs repeat
    bins = pd.qcut(p, q=n_bins, duplicates="drop")

    out = (
        pd.DataFrame({"bin": bins, "y": y, "p": p})
        .groupby("bin", observed=True)
        .agg(
            n=("y", "size"),
            p_mean=("p", "mean"),   # predicted probability avg
            y_rate=("y", "mean"),   # observed frequency
            p_min=("p", "min"),
            p_max=("p", "max"),
        )
        .reset_index(drop=True)
        .sort_values("p_mean")
    )
    return out

# example: get calibration table for ONE group
#key = (10, "xgboost", 6, "daily")  # change
g = perf_df.set_index(gcols).loc[key].reset_index()
calib_tbl = calibration_table(g, n_bins=20)
round(calib_tbl,2)

In [16]:
gcols

['horizon', 'model', 'train_years', 'feature_set']